# 제5장: AI 정책 설계

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/back2zion/ai-governance-handbook/blob/main/notebooks/ch05a.ipynb)

> **『AI 거버넌스 실무 지침서』** (곽두일) 실습 코드
>
> 이 노트북의 모든 코드는 Google Colab에서 바로 실행할 수 있습니다.

In [ ]:
# 필요 패키지 설치 (최초 1회)
!pip install -q fairlearn shap mlflow diffprivlib alibi alibi-detect evidently pandas scikit-learn matplotlib seaborn numpy

**AI 영향평가 자동화 도구**

In [ ]:
from dataclasses import dataclass, field
from enum import Enum
from typing import Dict, List, Optional
from datetime import datetime


class RiskLevel(Enum):
    """EU AI Act 기반 위험 수준 분류"""
    UNACCEPTABLE = 4  # 금지
    HIGH = 3          # 고위험
    LIMITED = 2       # 제한 위험
    MINIMAL = 1       # 최소 위험


@dataclass
class AISystemProfile:
    """AI 시스템 프로파일"""
    system_id: str
    name: str
    purpose: str
    domain: str
    data_types: List[str]
    affected_population: List[str]
    automation_level: str  # full, partial, advisory
    decision_impact: str   # critical, significant, moderate, low


@dataclass
class ImpactAssessmentResult:
    """영향평가 결과"""
    system_id: str
    risk_level: RiskLevel
    scores: Dict[str, float]
    findings: List[str]
    mitigations: List[str]
    assessed_at: str = field(
        default_factory=lambda: datetime.now().isoformat()
    )


class AIImpactAssessor:
    """AI 영향평가 자동화 엔진"""

    # 고위험 도메인 (EU AI Act Annex III 기반)
    HIGH_RISK_DOMAINS = [
        "employment", "credit_scoring", "education",
        "law_enforcement", "immigration", "healthcare",
        "critical_infrastructure", "judiciary"
    ]

    # 민감 데이터 유형
    SENSITIVE_DATA_TYPES = [
        "biometric", "health", "financial",
        "criminal", "political", "ethnic", "religious"
    ]

    def pre_screening(
        self, profile: AISystemProfile
    ) -> RiskLevel:
        """1단계: 사전 스크리닝"""
        score = 0

        # 도메인 기반 평가
        if profile.domain in self.HIGH_RISK_DOMAINS:
            score += 3

        # 민감 데이터 사용 여부
        sensitive_count = sum(
            1 for dt in profile.data_types
            if dt in self.SENSITIVE_DATA_TYPES
        )
        score += min(sensitive_count, 3)

        # 자동화 수준
        automation_scores = {
            "full": 3, "partial": 2, "advisory": 1
        }
        score += automation_scores.get(
            profile.automation_level, 0
        )

        # 의사결정 영향도
        impact_scores = {
            "critical": 3, "significant": 2,
            "moderate": 1, "low": 0
        }
        score += impact_scores.get(
            profile.decision_impact, 0
        )

        # 위험 수준 분류
        if score >= 10:
            return RiskLevel.UNACCEPTABLE
        elif score >= 7:
            return RiskLevel.HIGH
        elif score >= 4:
            return RiskLevel.LIMITED
        else:
            return RiskLevel.MINIMAL

    def detailed_assessment(
        self,
        profile: AISystemProfile,
        fairness_score: float,
        transparency_score: float,
        safety_score: float,
        privacy_score: float,
        accountability_score: float
    ) -> ImpactAssessmentResult:
        """2단계: 상세 영향평가"""
        risk_level = self.pre_screening(profile)

        scores = {
            "fairness": fairness_score,
            "transparency": transparency_score,
            "safety": safety_score,
            "privacy": privacy_score,
            "accountability": accountability_score,
        }
        avg = sum(scores.values()) / len(scores)
        scores["overall"] = round(avg, 2)

        findings = []
        mitigations = []

        if fairness_score < 0.7:
            findings.append(
                "공정성 점수 미달: 편향 위험 존재"
            )
            mitigations.append(
                "편향 탐지 및 완화 기법 적용 필요"
            )
        if transparency_score < 0.6:
            findings.append(
                "투명성 부족: 설명가능성 개선 필요"
            )
            mitigations.append(
                "SHAP/LIME 기반 설명 모듈 추가"
            )
        if privacy_score < 0.7:
            findings.append(
                "프라이버시 위험: 데이터 보호 강화 필요"
            )
            mitigations.append(
                "차분 프라이버시 또는 연합학습 적용"
            )

        return ImpactAssessmentResult(
            system_id=profile.system_id,
            risk_level=risk_level,
            scores=scores,
            findings=findings,
            mitigations=mitigations,
        )

### 실행

In [ ]:
if __name__ == "__main__":
    assessor = AIImpactAssessor()

    profile = AISystemProfile(
        system_id="AI-2026-001",
        name="AI 채용 심사 시스템",
        purpose="이력서 자동 스크리닝 및 적합도 평가",
        domain="employment",
        data_types=["biometric", "ethnic"],
        affected_population=["job_applicants"],
        automation_level="partial",
        decision_impact="significant",
    )

    result = assessor.detailed_assessment(
        profile,
        fairness_score=0.65,
        transparency_score=0.55,
        safety_score=0.85,
        privacy_score=0.60,
        accountability_score=0.80,
    )

    print(f"위험 수준: {result.risk_level.name}")
    print(f"종합 점수: {result.scores['overall']}")
    for f in result.findings:
        print(f"  발견사항: {f}")
    for m in result.mitigations:
        print(f"  완화 조치: {m}")

**NIST AI RMF 기반 리스크 자동 평가 엔진**

In [ ]:
from dataclasses import dataclass, field
from typing import Dict, List, Tuple
from enum import Enum
import json


class RiskGrade(Enum):
    CRITICAL = "critical"
    HIGH = "high"
    MEDIUM = "medium"
    LOW = "low"


class ResponseStrategy(Enum):
    TREAT = "treat"
    TRANSFER = "transfer"
    TERMINATE = "terminate"
    TOLERATE = "tolerate"


@dataclass
class RiskItem:
    """개별 리스크 항목"""
    risk_id: str
    category: str       # technical, operational, compliance,
                        # strategic, ethical
    description: str
    likelihood: int      # 1-5
    impact: int          # 1-5
    owner: str
    strategy: str = ""
    mitigations: List[str] = field(default_factory=list)

    @property
    def score(self) -> int:
        return self.likelihood * self.impact

    @property
    def grade(self) -> RiskGrade:
        s = self.score
        if s >= 20:
            return RiskGrade.CRITICAL
        elif s >= 12:
            return RiskGrade.HIGH
        elif s >= 6:
            return RiskGrade.MEDIUM
        else:
            return RiskGrade.LOW


class NISTRiskAssessmentEngine:
    """NIST AI RMF 4대 기능 기반 리스크 평가 엔진"""

    def __init__(self, org_risk_appetite: int = 12):
        self.risk_appetite = org_risk_appetite
        self.risk_register: List[RiskItem] = []

    # -- Govern 기능 --
    def set_risk_appetite(self, level: int):
        """조직의 리스크 허용 수준 설정"""
        self.risk_appetite = level

    # -- Map 기능 --
    def identify_risks(
        self, system_id: str, system_info: Dict
    ) -> List[RiskItem]:
        """AI 시스템 맥락 기반 리스크 자동 식별"""
        risks = []

        # 기술 리스크 점검
        if system_info.get("model_type") == "deep_learning":
            risks.append(RiskItem(
                risk_id=f"{system_id}-T001",
                category="technical",
                description="딥러닝 모델 적대적 공격 취약성",
                likelihood=3, impact=4,
                owner=system_info.get("owner", ""),
            ))
        if system_info.get("input_type") == "user_text":
            risks.append(RiskItem(
                risk_id=f"{system_id}-T002",
                category="technical",
                description="프롬프트 인젝션 공격 위험",
                likelihood=4, impact=3,
                owner=system_info.get("owner", ""),
            ))

        # 윤리 리스크 점검
        if system_info.get("affects_individuals", False):
            risks.append(RiskItem(
                risk_id=f"{system_id}-E001",
                category="ethical",
                description="개인 대상 의사결정 편향 위험",
                likelihood=3, impact=4,
                owner=system_info.get("owner", ""),
            ))

        # 컴플라이언스 리스크 점검
        if system_info.get("processes_personal_data", False):
            risks.append(RiskItem(
                risk_id=f"{system_id}-C001",
                category="compliance",
                description="개인정보보호법 위반 위험",
                likelihood=2, impact=5,
                owner=system_info.get("owner", ""),
            ))

        # 운영 리스크 점검
        if system_info.get("is_production", False):
            risks.append(RiskItem(
                risk_id=f"{system_id}-O001",
                category="operational",
                description="데이터 드리프트로 인한 성능 저하",
                likelihood=4, impact=3,
                owner=system_info.get("owner", ""),
            ))

        return risks

    # -- Measure 기능 --
    def evaluate_risks(
        self, risks: List[RiskItem]
    ) -> Dict[str, object]:
        """리스크 정량 평가 및 등급 산정"""
        results = {
            "total_risks": len(risks),
            "by_grade": {g.value: 0 for g in RiskGrade},
            "by_category": {},
            "above_appetite": [],
            "risk_details": [],
        }

        for risk in risks:
            grade = risk.grade.value
            results["by_grade"][grade] += 1

            cat = risk.category
            if cat not in results["by_category"]:
                results["by_category"][cat] = []
            results["by_category"][cat].append(risk.risk_id)

            if risk.score > self.risk_appetite:
                results["above_appetite"].append({
                    "id": risk.risk_id,
                    "score": risk.score,
                    "grade": grade,
                })

            results["risk_details"].append({
                "id": risk.risk_id,
                "score": risk.score,
                "grade": grade,
                "category": cat,
            })

        return results

    # -- Manage 기능 --
    def recommend_strategy(
        self, risk: RiskItem
    ) -> ResponseStrategy:
        """4T 모델 기반 대응 전략 추천"""
        if risk.score >= 20:
            return ResponseStrategy.TERMINATE
        elif risk.score >= 12:
            return ResponseStrategy.TREAT
        elif risk.score >= 6:
            if risk.category == "compliance":
                return ResponseStrategy.TREAT
            return ResponseStrategy.TRANSFER
        else:
            return ResponseStrategy.TOLERATE

    def generate_report(
        self, system_id: str, risks: List[RiskItem]
    ) -> str:
        """리스크 평가 보고서 생성"""
        evaluation = self.evaluate_risks(risks)
        lines = [
            f"=== 리스크 평가 보고서: {system_id} ===",
            f"총 리스크 수: {evaluation['total_risks']}",
            f"등급별 분포: {evaluation['by_grade']}",
            f"허용 수준 초과: "
            f"{len(evaluation['above_appetite'])}건",
            "",
        ]
        for risk in risks:
            strategy = self.recommend_strategy(risk)
            lines.append(
                f"[{risk.risk_id}] 점수={risk.score} "
                f"등급={risk.grade.value} "
                f"전략={strategy.value}"
            )
        return "\n".join(lines)

### 실행

In [ ]:
if __name__ == "__main__":
    engine = NISTRiskAssessmentEngine(org_risk_appetite=12)

    system_info = {
        "model_type": "deep_learning",
        "input_type": "user_text",
        "affects_individuals": True,
        "processes_personal_data": True,
        "is_production": True,
        "owner": "AI개발팀",
    }

    risks = engine.identify_risks("SYS-001", system_info)
    report = engine.generate_report("SYS-001", risks)
    print(report)

**Policy-as-Code 종합 구현: 배포 게이트**

In [ ]:
from dataclasses import dataclass, field
from typing import Dict, List, Optional
from enum import Enum
from datetime import datetime
import json


class GateDecision(Enum):
    APPROVED = "approved"
    REJECTED = "rejected"
    CONDITIONAL = "conditional"


@dataclass
class PolicyRule:
    """개별 정책 규칙"""
    rule_id: str
    name: str
    category: str
    threshold: float
    operator: str  # gte, lte, eq, exists
    description: str


@dataclass
class CheckResult:
    """개별 검증 결과"""
    rule_id: str
    passed: bool
    actual_value: object
    expected: str
    message: str


@dataclass
class GateResult:
    """배포 게이트 종합 결과"""
    model_id: str
    decision: GateDecision
    checks: List[CheckResult]
    timestamp: str = field(
        default_factory=lambda: datetime.now().isoformat()
    )

    @property
    def pass_rate(self) -> float:
        if not self.checks:
            return 0.0
        passed = sum(1 for c in self.checks if c.passed)
        return round(passed / len(self.checks), 3)


class AIDeploymentGate:
    """AI 모델 배포 게이트 - Policy-as-Code 엔진"""

    def __init__(self):
        self.rules: List[PolicyRule] = []
        self._load_default_rules()

    def _load_default_rules(self):
        """기본 정책 규칙 로드"""
        self.rules = [
            # 성능 기준
            PolicyRule(
                "PERF-001", "최소 정확도",
                "performance", 0.85, "gte",
                "모델 정확도 85% 이상"
            ),
            PolicyRule(
                "PERF-002", "최소 F1 점수",
                "performance", 0.80, "gte",
                "F1 점수 80% 이상"
            ),
            # 공정성 기준
            PolicyRule(
                "FAIR-001", "인구통계 패리티 비율",
                "fairness", 0.80, "gte",
                "DP 비율 0.80 이상"
            ),
            PolicyRule(
                "FAIR-002", "균등 기회 차이",
                "fairness", 0.10, "lte",
                "균등 기회 차이 0.10 이하"
            ),
            # 프라이버시 기준
            PolicyRule(
                "PRIV-001", "차분 프라이버시 엡실론",
                "privacy", 1.0, "lte",
                "DP 엡실론 1.0 이하"
            ),
            PolicyRule(
                "PRIV-002", "비식별화 적용",
                "privacy", 1.0, "eq",
                "비식별화 기법 적용 여부"
            ),
            # 문서 기준
            PolicyRule(
                "DOC-001", "모델 카드 존재",
                "documentation", 1.0, "exists",
                "모델 카드 작성 완료"
            ),
            PolicyRule(
                "DOC-002", "영향평가 완료",
                "documentation", 1.0, "exists",
                "영향평가 보고서 존재"
            ),
        ]

    def _evaluate_rule(
        self, rule: PolicyRule, value: object
    ) -> CheckResult:
        """개별 규칙 평가"""
        passed = False
        if rule.operator == "gte":
            passed = float(value) >= rule.threshold
        elif rule.operator == "lte":
            passed = float(value) <= rule.threshold
        elif rule.operator == "eq":
            passed = float(value) == rule.threshold
        elif rule.operator == "exists":
            passed = value is not None and value != ""

        expected = (
            f"{rule.operator} {rule.threshold}"
        )
        return CheckResult(
            rule_id=rule.rule_id,
            passed=passed,
            actual_value=value,
            expected=expected,
            message=rule.description,
        )

    def evaluate(
        self, model_id: str, metrics: Dict
    ) -> GateResult:
        """배포 게이트 종합 평가"""
        checks = []
        metric_map = {
            "PERF-001": metrics.get("accuracy"),
            "PERF-002": metrics.get("f1_score"),
            "FAIR-001": metrics.get("dp_ratio"),
            "FAIR-002": metrics.get("eq_opportunity_diff"),
            "PRIV-001": metrics.get("dp_epsilon"),
            "PRIV-002": metrics.get("anonymized"),
            "DOC-001": metrics.get("model_card"),
            "DOC-002": metrics.get("impact_assessment"),
        }

        for rule in self.rules:
            value = metric_map.get(rule.rule_id)
            result = self._evaluate_rule(rule, value)
            checks.append(result)

        # 의사결정 로직
        failed = [c for c in checks if not c.passed]
        critical_fail = any(
            c.rule_id.startswith(("FAIR", "PRIV"))
            for c in failed
        )

        if not failed:
            decision = GateDecision.APPROVED
        elif critical_fail:
            decision = GateDecision.REJECTED
        else:
            decision = GateDecision.CONDITIONAL

        return GateResult(
            model_id=model_id,
            decision=decision,
            checks=checks,
        )

    def generate_report(self, result: GateResult) -> str:
        """배포 게이트 결과 보고서 생성"""
        lines = [
            "=" * 50,
            f"배포 게이트 결과: {result.model_id}",
            f"판정: {result.decision.value}",
            f"통과율: {result.pass_rate * 100:.1f}%",
            f"평가 시점: {result.timestamp}",
            "-" * 50,
        ]
        for c in result.checks:
            status = "PASS" if c.passed else "FAIL"
            lines.append(
                f"[{status}] {c.rule_id}: "
                f"{c.message} "
                f"(실제: {c.actual_value}, "
                f"기준: {c.expected})"
            )
        lines.append("=" * 50)
        return "\n".join(lines)

### 실행

In [ ]:
if __name__ == "__main__":
    gate = AIDeploymentGate()

    metrics = {
        "accuracy": 0.92,
        "f1_score": 0.88,
        "dp_ratio": 0.78,
        "eq_opportunity_diff": 0.05,
        "dp_epsilon": 0.8,
        "anonymized": 1.0,
        "model_card": "model_card_v2.md",
        "impact_assessment": "ia_report_2026.pdf",
    }

    result = gate.evaluate("MODEL-2026-042", metrics)
    print(gate.generate_report(result))

**AI 인시던트 관리 시스템**

In [ ]:
from dataclasses import dataclass, field
from typing import Dict, List, Optional
from enum import Enum
from datetime import datetime


class Severity(Enum):
    CRITICAL = 1    # 치명적
    MAJOR = 2       # 심각
    MODERATE = 3    # 보통
    MINOR = 4       # 경미


class IncidentStatus(Enum):
    REPORTED = "reported"
    TRIAGED = "triaged"
    CONTAINED = "contained"
    INVESTIGATING = "investigating"
    RESOLVED = "resolved"
    CLOSED = "closed"


class IncidentType(Enum):
    PERFORMANCE_DEGRADATION = "performance_degradation"
    BIAS_DETECTED = "bias_detected"
    PROMPT_INJECTION = "prompt_injection"
    HALLUCINATION = "hallucination"
    DATA_BREACH = "data_breach"
    ADVERSARIAL_ATTACK = "adversarial_attack"
    DATA_DRIFT = "data_drift"
    SERVICE_OUTAGE = "service_outage"


@dataclass

### 실행

In [ ]:
class Incident:
    """AI 인시던트"""
    incident_id: str
    title: str
    incident_type: IncidentType
    severity: Severity
    system_id: str
    description: str
    reported_by: str
    reported_at: str = field(
        default_factory=lambda: datetime.now().isoformat()
    )
    status: IncidentStatus = IncidentStatus.REPORTED
    assigned_to: str = ""
    timeline: List[Dict] = field(default_factory=list)
    root_cause: str = ""
    resolution: str = ""
    lessons_learned: List[str] = field(
        default_factory=list
    )

    def add_timeline_entry(
        self, action: str, actor: str
    ):
        self.timeline.append({
            "timestamp": datetime.now().isoformat(),
            "action": action,
            "actor": actor,
        })


# 에스컬레이션 규칙

### 실행

In [ ]:
ESCALATION_RULES = {
    Severity.CRITICAL: {
        "response_time_minutes": 15,
        "notify": [
            "ceo", "cto", "legal",
            "governance_committee"
        ],
        "actions": [
            "시스템 즉시 중단",
            "인시던트 대응팀 긴급 소집",
            "규제 기관 통보 준비",
        ],
    },
    Severity.MAJOR: {
        "response_time_minutes": 60,
        "notify": ["cto", "governance_team"],
        "actions": [
            "인적 감독 전환",
            "영향 범위 긴급 파악",
            "사용자 고지 준비",
        ],
    },
    Severity.MODERATE: {
        "response_time_minutes": 240,
        "notify": ["team_lead", "governance_team"],
        "actions": [
            "모니터링 강화",
            "원인 분석 착수",
            "임시 완화 조치",
        ],
    },
    Severity.MINOR: {
        "response_time_minutes": 1440,
        "notify": ["team_lead"],
        "actions": [
            "일반 대응 절차",
            "다음 정기 검토에 포함",
        ],
    },
}


class AIIncidentManager:
    """AI 인시던트 관리 시스템"""

    def __init__(self):
        self.incidents: Dict[str, Incident] = {}
        self._counter = 0

    def report_incident(
        self,
        title: str,
        incident_type: IncidentType,
        severity: Severity,
        system_id: str,
        description: str,
        reported_by: str,
    ) -> Incident:
        """인시던트 등록"""
        self._counter += 1
        incident_id = (
            f"INC-{datetime.now().year}-"
            f"{self._counter:04d}"
        )

        incident = Incident(
            incident_id=incident_id,
            title=title,
            incident_type=incident_type,
            severity=severity,
            system_id=system_id,
            description=description,
            reported_by=reported_by,
        )
        incident.add_timeline_entry(
            "인시던트 등록", reported_by
        )

        self.incidents[incident_id] = incident

        # 자동 에스컬레이션
        self._escalate(incident)

        return incident

    def _escalate(self, incident: Incident):
        """에스컬레이션 규칙 적용"""
        rules = ESCALATION_RULES[incident.severity]
        incident.add_timeline_entry(
            f"에스컬레이션: "
            f"{rules['notify']}에 통보, "
            f"대응 시한 "
            f"{rules['response_time_minutes']}분",
            "system",
        )
        for action in rules["actions"]:
            incident.add_timeline_entry(
                f"권고 조치: {action}", "system"
            )

    def triage(
        self,
        incident_id: str,
        assigned_to: str,
        updated_severity: Optional[Severity] = None,
    ):
        """인시던트 분류"""
        inc = self.incidents[incident_id]
        inc.status = IncidentStatus.TRIAGED
        inc.assigned_to = assigned_to
        if updated_severity:
            inc.severity = updated_severity
            self._escalate(inc)
        inc.add_timeline_entry(
            f"{assigned_to}에게 할당", "triage_team"
        )

    def contain(
        self, incident_id: str, actions: List[str]
    ):
        """봉쇄 조치"""
        inc = self.incidents[incident_id]
        inc.status = IncidentStatus.CONTAINED
        for action in actions:
            inc.add_timeline_entry(
                f"봉쇄 조치: {action}", inc.assigned_to
            )

    def resolve(
        self,
        incident_id: str,
        root_cause: str,
        resolution: str,
        lessons: List[str],
    ):
        """인시던트 해결"""
        inc = self.incidents[incident_id]
        inc.status = IncidentStatus.RESOLVED
        inc.root_cause = root_cause
        inc.resolution = resolution
        inc.lessons_learned = lessons
        inc.add_timeline_entry(
            "인시던트 해결 완료", inc.assigned_to
        )

    def generate_postmortem(
        self, incident_id: str
    ) -> str:
        """사후 검토 보고서 생성"""
        inc = self.incidents[incident_id]
        lines = [
            "=" * 50,
            f"사후 검토 보고서: {inc.incident_id}",
            "=" * 50,
            f"제목: {inc.title}",
            f"유형: {inc.incident_type.value}",
            f"심각도: {inc.severity.name}",
            f"시스템: {inc.system_id}",
            f"보고자: {inc.reported_by}",
            f"보고 시각: {inc.reported_at}",
            f"상태: {inc.status.value}",
            "",
            "--- 근본 원인 ---",
            inc.root_cause or "분석 중",
            "",
            "--- 해결 조치 ---",
            inc.resolution or "진행 중",
            "",
            "--- 교훈 ---",
        ]
        for lesson in inc.lessons_learned:
            lines.append(f"  - {lesson}")
        lines.append("")
        lines.append("--- 타임라인 ---")
        for entry in inc.timeline:
            lines.append(
                f"  [{entry['timestamp']}] "
                f"{entry['action']} "
                f"(by {entry['actor']})"
            )
        return "\n".join(lines)

    def get_dashboard_data(self) -> Dict:
        """대시보드용 데이터 산출"""
        data = {
            "total": len(self.incidents),
            "by_severity": {},
            "by_status": {},
            "by_type": {},
            "open_incidents": [],
        }
        for inc in self.incidents.values():
            sev = inc.severity.name
            data["by_severity"][sev] = (
                data["by_severity"].get(sev, 0) + 1
            )
            st = inc.status.value
            data["by_status"][st] = (
                data["by_status"].get(st, 0) + 1
            )
            tp = inc.incident_type.value
            data["by_type"][tp] = (
                data["by_type"].get(tp, 0) + 1
            )
            if inc.status not in (
                IncidentStatus.RESOLVED,
                IncidentStatus.CLOSED
            ):
                data["open_incidents"].append(
                    inc.incident_id
                )
        return data

### 실행

In [ ]:
# 사용 예시
if __name__ == "__main__":
    manager = AIIncidentManager()

    # 인시던트 등록
    inc = manager.report_incident(
        title="채용 AI 성별 편향 탐지",
        incident_type=IncidentType.BIAS_DETECTED,
        severity=Severity.MAJOR,
        system_id="SYS-RECRUIT-001",
        description=(
            "채용 심사 AI에서 여성 지원자 합격률이 "
            "남성 대비 65% 수준으로 하락"
        ),
        reported_by="monitoring_system",
    )

    # 분류
    manager.triage(inc.incident_id, "AI거버넌스팀")

    # 봉쇄
    manager.contain(inc.incident_id, [
        "해당 AI 시스템 일시 중단",
        "수동 심사로 전환",
        "영향 받은 지원자 목록 확보",
    ])

    # 해결
    manager.resolve(
        inc.incident_id,
        root_cause=(
            "학습 데이터 내 역사적 채용 편향 반영"
        ),
        resolution=(
            "편향 완화 기법 적용 후 재학습, "
            "공정성 메트릭 임계값 강화"
        ),
        lessons=[
            "학습 데이터 편향 사전 점검 절차 추가",
            "공정성 모니터링 임계값 조정",
            "분기별 편향 감사 의무화",
        ],
    )

    print(manager.generate_postmortem(inc.incident_id))
    print("\n대시보드:", manager.get_dashboard_data())